# vlapol: fine-tune SmolVLA on MuJoCo pick-and-place (Colab, free T4)

Fine-tunes `lerobot/smolvla_base` on scripted demonstrations of a floating gripper with a wrist camera, then evaluates the policy closed loop in the same MuJoCo sim.

**Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.

Everything that must survive a disconnect lives on Google Drive under `MyDrive/vlapol/`: the dataset, the two newest checkpoints and the eval results. After a disconnect, run the cells again from the top: the dataset is reused and training resumes from the newest checkpoint.

## 1. Setup

In [ ]:
# Settings
REPO_URL = "https://github.com/teiband/vlapol.git"
BRANCH = "main"            # branch that contains the vlapol package
RUN_NAME = "smolvla_pickplace"
NUM_EPISODES = 100         # demonstrations to collect
CAMERAS = "wrist"          # "wrist" or "wrist scene" (adds a fixed overview camera)
STEPS = 20_000             # total training steps
BATCH_SIZE = 16
SAVE_FREQ = 2_000          # checkpoint interval (steps)
EVAL_EPISODES = 50

In [ ]:
import os, subprocess
from pathlib import Path
from google.colab import drive

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "No GPU: select a T4 runtime.")
drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/vlapol")
DRIVE_DATASET = DRIVE_DIR / "datasets" / "vlapol_pickplace"
RUN_DIR = DRIVE_DIR / "runs" / RUN_NAME
LOCAL_DATASET = Path("/content/data/vlapol_pickplace")
LOCAL_RUN = Path("/content/outputs") / RUN_NAME
os.environ["MUJOCO_GL"] = "egl"   # offscreen GPU rendering

If the repository is private, cloning asks for credentials: use `https://<token>@github.com/teiband/vlapol.git` as `REPO_URL` with a GitHub personal access token that has read access.

In [ ]:
%cd /content
!test -d vlapol || git clone --branch {BRANCH} {REPO_URL} vlapol
%cd /content/vlapol
!git pull --ff-only
!pip install -q -e ".[lerobot]"

# The editable install is only picked up by new Python processes. Make the package
# importable in this already-running kernel too, and drop any half-imported copy.
import os, sys
SRC = "/content/vlapol/src"
assert os.path.isfile(f"{SRC}/vlapol/env.py"), f"{SRC}/vlapol/env.py is missing: did the clone above succeed?"
if SRC not in sys.path:
    sys.path.insert(0, SRC)
for name in [m for m in sys.modules if m == "vlapol" or m.startswith("vlapol.")]:
    del sys.modules[name]
import vlapol.env
print("vlapol imported from", vlapol.__file__)

## 2. Check the simulation

Runs the scripted expert for a few episodes (it should succeed every time) and shows the wrist-camera view next to an overview.

In [ ]:
!python -m vlapol.evaluate --expert --episodes 5 --videos 0

In [ ]:
import numpy as np
from PIL import Image
from vlapol.env import PickPlaceEnv

env = PickPlaceEnv(cameras=("wrist", "scene"))
obs = env.reset(seed=0)
print(env.task)
display(Image.fromarray(np.concatenate([obs["images"]["wrist"], obs["images"]["scene"]], axis=1)))
env.close()

## 3. Collect demonstrations

Writes a `LeRobotDataset` (v3.0) with one task string per episode. The dataset is built on the local disk and then copied to Drive; if Drive already has it, it is copied back instead of being collected again.

In [ ]:
import json, shutil

def n_episodes(root):
    info = Path(root) / "meta" / "info.json"
    return json.loads(info.read_text())["total_episodes"] if info.is_file() else 0

if n_episodes(DRIVE_DATASET) > n_episodes(LOCAL_DATASET):
    shutil.rmtree(LOCAL_DATASET, ignore_errors=True)
    shutil.copytree(DRIVE_DATASET, LOCAL_DATASET)
    print(f"restored {n_episodes(LOCAL_DATASET)} episodes from Drive")

if n_episodes(LOCAL_DATASET) < NUM_EPISODES:
    !python -m vlapol.collect --root {LOCAL_DATASET} --episodes {NUM_EPISODES} --cameras {CAMERAS} --resume 2>&1 | grep -E "episode|dataset|Error|Traceback"
    shutil.rmtree(DRIVE_DATASET, ignore_errors=True)
    shutil.copytree(LOCAL_DATASET, DRIVE_DATASET)
print(f"dataset ready: {n_episodes(LOCAL_DATASET)} episodes at {LOCAL_DATASET} (copy on Drive: {DRIVE_DATASET})")

## 4. Fine-tune SmolVLA

`vlapol.train` wraps `lerobot-train`. It trains on the local disk and copies each finished checkpoint to Drive, keeping the two newest. **Re-running this cell resumes from the newest checkpoint**; it does nothing once `STEPS` is reached. To train longer, raise `STEPS` and run it again.

Extra `lerobot-train` flags can be appended to the command, for example `--accelerator.mixed_precision=fp16` to try mixed precision on the T4.

Deleted checkpoints go to the Drive trash and still count against your quota until the trash is emptied.

In [ ]:
!python -m vlapol.train \
    --dataset-root {LOCAL_DATASET} \
    --run-dir {RUN_DIR} \
    --local-dir {LOCAL_RUN} \
    --steps {STEPS} --batch-size {BATCH_SIZE} --save-freq {SAVE_FREQ} \
    --policy.device=cuda

## 5. Closed-loop evaluation

Rolls out the newest checkpoint on unseen cube/target layouts. `eval.json` and a few rollout videos (overview + wrist view) are written to Drive.

In [ ]:
from vlapol.train import latest_checkpoint

ckpt = latest_checkpoint(RUN_DIR)
assert ckpt is not None, f"no checkpoint in {RUN_DIR} yet"
EVAL_DIR = RUN_DIR / "eval" / ckpt.name
print("evaluating", ckpt)
!python -m vlapol.evaluate --policy-path {ckpt}/pretrained_model --episodes {EVAL_EPISODES} --output-dir {EVAL_DIR} 2>&1 | grep -E "episode|success rate|Error|Traceback"

In [ ]:
from IPython.display import Video

videos = sorted(EVAL_DIR.glob("*.mp4"))
print([v.name for v in videos])
Video(str(videos[0]), embed=True) if videos else None